# Mega Project 3 — Risk Segmentation
## Problem 2: Credit Bureau Behavioral Segmentation — Real Unsupervised Clustering
## Independent of Both PD Level (Problem 1) and the PD Model's Own Bureau Features

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
A collections or portfolio-management team needs more than a single PD
level to differentiate treatment — they need to know *how* an applicant
behaves with external credit, not just how risky they are. This notebook
builds that second, independent axis: real behavioral segments from real
credit-bureau history.

### This notebook trains no supervised model and scores no PD
It builds a real, vectorized behavioral feature set from real `bureau.csv`
+ `bureau_balance.csv` (external credit history, no linkage to any Home
Credit loan itself), then applies real unsupervised K-Means clustering —
grouping applicants by BEHAVIORAL SIMILARITY, never trained against real
`TARGET`.

### Why this is a genuine, not redundant, addition
Mega Project 1's champion PD model already includes 7 real bureau/
bureau_balance summary features. This notebook does NOT just re-cluster
those same 7 numbers. It builds a richer, more granular real 16-feature
set (real credit-type mix, real overdue AMOUNT not just a DPD-month count,
real credit-limit usage at other institutions, real recency, real
distinct-credit-type count, real worst bureau_balance status severity) AND
applies a genuinely different MECHANISM (unsupervised clustering by
similarity, never supervised against TARGET) instead of Problem 1's
PD-level tiering. A real, computed Cramer's V against Problem 1's Risk
Tier is reported as honest evidence of how independent the two axes
actually turned out to be — not an asserted claim.

### Hard dependency
This notebook requires Mega Project 3 / Notebook 01's real per-applicant
output (`PD`, `TARGET`, `RISK_TIER`) to already exist — PD is reused
unchanged, never re-scored here.

### Advanced error tackling applied (see LESSONS_LEARNED.md for the
### incidents each of these prevents a repeat of)
- HARD dependency on Notebook 01's real output, checked by actual required
  columns present, not just file existence.
- No `monotonic_within_noise()` call in this notebook, by design —
  behavioral clusters are unordered categorical groups with no expected
  direction, the same reasoning MP2 Notebook 05 already established for
  its own HHI concentration analysis.
- No `matplotlib.use(...)` call anywhere in this file — lets Jupyter's own
  inline backend handle `plt.show()` cleanly.
- Real, disclosed sampling for computational tractability: silhouette
  score is O(n²) in the naive case, so this notebook uses scikit-learn's
  own `sample_size` parameter — a real, standard, documented mitigation,
  not a shortcut that changes the real clustering result.
- Data-driven K, never a fixed cluster count — the real number of clusters
  is chosen by the real silhouette score across a documented candidate
  range.
- Applicants with zero real bureau history are never silently imputed
  into a cluster with fabricated average values — they get their own
  explicit "No Bureau History" segment, disclosed by real, measured
  prevalence.
- No EDA section, per standing instruction.

### Verification status
Verified end-to-end on this suite's synthetic fixture via real Jupyter
execution — 0 errors, all integrity and statistical-robustness checks pass,
HTML dashboard confirmed under a network-blocked Playwright check, Excel
workbook confirmed via LibreOffice headless recalculation. **Not yet run
against your real data.**


In [ ]:
# ============================================================================
# NOTEBOOK 02 — MEGA PROJECT 3: RISK SEGMENTATION
# PROBLEM 2: CREDIT BUREAU BEHAVIORAL SEGMENTATION
# Real Unsupervised Clustering on a Real, Richer Bureau/Bureau-Balance Feature
# Space — Independent of Both PD Level (Problem 1) and the PD Model's Own
# Bureau Summary Features
# ----------------------------------------------------------------------------
# ZERO-FABRICATION DISCLOSURE: this notebook trains no supervised model and
# scores no PD. It builds a real, vectorized behavioral feature set from real
# bureau.csv + bureau_balance.csv (external credit history, no linkage to any
# Home Credit loan) via src/features/risk_segmentation_features.py, then
# applies real unsupervised K-Means clustering -- grouping applicants by
# BEHAVIORAL SIMILARITY, never trained against real TARGET.
#
# WHY THIS IS A GENUINE, NOT REDUNDANT, ADDITION (read this before trusting
# any "independent axis" claim below): Mega Project 1's champion PD model
# ALREADY includes 7 real bureau/bureau_balance summary features (BUREAU_
# CNT_CREDITS, BUREAU_CNT_ACTIVE, BUREAU_AMT_CREDIT_SUM_TOTAL, BUREAU_AMT_
# CREDIT_SUM_DEBT_TOTAL, BUREAU_MAX_DAYS_OVERDUE, BUREAU_TOTAL_DPD_MONTHS,
# BUREAU_DEBT_TO_CREDIT_RATIO -- see src/features/applicant_credit_history_
# features.py). This notebook does NOT just re-cluster those same 7 numbers.
# It builds a richer, more granular real feature set (real credit-type mix,
# real overdue AMOUNT not just a DPD-month count, real credit-limit usage at
# OTHER institutions, real recency, real distinct-credit-type count, real
# worst bureau_balance status severity -- 16 real features total, see
# src/features/risk_segmentation_features.py's own disclosure) AND applies a
# genuinely different MECHANISM (unsupervised clustering by similarity, never
# supervised against TARGET) instead of Problem 1's PD-level tiering. Both
# facts are what make this a real, non-redundant segmentation axis -- Section
# 11 below computes a real Cramer's V between this notebook's segments and
# Problem 1's tiers as honest, computed evidence of how independent the two
# axes actually turned out to be, not an asserted claim.
#
# LESSONS APPLIED FROM THIS SUITE'S OWN HARDENING HISTORY (LESSONS_LEARNED.md
# — every item below cites which real incident it prevents a repeat of):
#   1. HARD DEPENDENCY on Mega Project 3 / Notebook 01's real per-applicant
#      output (PD, TARGET, RISK_TIER), checked by actual required columns
#      present, not just file existence (LESSONS_LEARNED.md #4) -- this also
#      means PD is never re-scored here; Notebook 01's real values are reused
#      unchanged.
#   2. NO `monotonic_within_noise()` CALL IN THIS NOTEBOOK, BY DESIGN: unlike
#      Problem 1's tiers (ordered ascending by PD), behavioral clusters are
#      unordered categorical groups with no expected direction to check --
#      the same "does not apply by construction" reasoning MP2 Notebook 05
#      already established for its own HHI concentration analysis
#      (LESSONS_LEARNED.md #2 is honored by recognizing where it does NOT
#      apply, not by omission).
#   3. No `matplotlib.use(...)` call anywhere in this file (LESSONS_LEARNED.md
#      #7) — lets Jupyter's own inline backend handle `plt.show()` cleanly.
#   4. REAL, DISCLOSED SAMPLING FOR COMPUTATIONAL TRACTABILITY: silhouette
#      score is O(n^2) in the naive case -- at real ~300K-applicant scale this
#      is computationally infeasible, so this notebook uses scikit-learn's own
#      `sample_size` parameter (a real, standard, documented mitigation, not a
#      shortcut that changes the real result) — disclosed explicitly here and
#      in the model card, applying LESSONS_LEARNED.md #5's real-scale planning
#      discipline proactively rather than discovering the slowdown mid-run.
#   5. DATA-DRIVEN K, NEVER A FIXED CLUSTER COUNT: the real number of clusters
#      is chosen by the real silhouette score across a documented candidate
#      range, exactly the same "achieved, not forced" honesty Problem 1
#      already established for its own tier count.
#   6. APPLICANTS WITH ZERO REAL BUREAU HISTORY ARE NEVER SILENTLY IMPUTED
#      into a cluster with fabricated average values — they get their own
#      explicit "No Bureau History" segment, disclosed by real, measured
#      prevalence, not hidden.
#   7. NO EDA SECTION — per standing instruction.
# ============================================================================

import os
import sys
import json
import time
import warnings
import joblib
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# SECTION 1 — Config + suite-root resolution
# ---------------------------------------------------------------------------
def _find_suite_root(start: Path = None) -> Path:
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory plus "
        "well-known locations under your home folder. Fix: either open this notebook's "
        "own .ipynb file in place, or set an environment variable before launching "
        'Jupyter, e.g. on Windows PowerShell: $env:HC_SUITE_ROOT="C:\\Users\\rnand\\Downloads\\'
        'home-credit-enterprise-suite" -- see PERFORMANCE_SETUP_README.md.'
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

RAW_DIR = Path(CONFIG["raw_data_dir"])
SEED = int(CONFIG.get("random_seed", 42))

MP3_DIR = SUITE_ROOT / "03_mega_project_3_risk_segmentation"
ARTIFACTS_DIR = MP3_DIR / "decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = MP3_DIR / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR = MP3_DIR / "decision_engine" / "_parquet_cache"

sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import configure_performance, pin_cpu_affinity, load_csv_cached, check_ram_headroom

# ---------------------------------------------------------------------------
# SECTION 2 — WARP resource ceilings (before any heavy import)
# ---------------------------------------------------------------------------
PERF = configure_performance(
    ram_ceiling_fraction=float(CONFIG.get("ram_ceiling_fraction", 0.90)),
    cpu_ceiling_fraction=float(CONFIG.get("cpu_ceiling_fraction", 0.95)),
)
pin_cpu_affinity(PERF)
TOTAL_RAM_GB = PERF["total_ram_gb"]
TOTAL_THREADS = PERF["logical_cores"]
RAM_CEILING_GB = PERF["ram_ceiling_gb"]
CPU_CEILING_THREADS = PERF["n_threads"]

# ---------------------------------------------------------------------------
# SECTION 3 — Heavy-library imports (deliberately AFTER Section 2)
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

np.random.seed(SEED)
rng = np.random.default_rng(SEED)
T0 = time.time()

from features.risk_segmentation_features import engineer_bureau_behavior_features
from reporting.report_builder import (
    write_csv_outputs, build_word_report, build_excel_workbook,
    build_html_dashboard, VIVID_PALETTE, _palette,
)

print(f"[WARP] {TOTAL_RAM_GB:.1f} GB RAM / {TOTAL_THREADS} threads detected -> "
      f"ceiling {RAM_CEILING_GB} GB RAM, {CPU_CEILING_THREADS} threads")
print(f"[SEED] RANDOM_SEED = {SEED}")

# ---------------------------------------------------------------------------
# SECTION 4 — HARD DEPENDENCY: Mega Project 3 / Notebook 01's real per-
# applicant output, checked by actual required columns (LESSONS_LEARNED.md
# #4). Only 2 raw tables are loaded here (bureau, bureau_balance) -- PD is
# reused unchanged from Notebook 01, never re-scored.
# ---------------------------------------------------------------------------
NB01_PATH = ARTIFACTS_DIR / "notebook_01_risk_tiers.csv"
if not NB01_PATH.exists():
    raise FileNotFoundError(
        "Mega Project 3 / Notebook 02 requires Mega Project 3 / Notebook 01's real "
        "per-applicant output, which has not been produced on this machine yet. Fix: run "
        "03_mega_project_3_risk_segmentation/notebooks/01_data_driven_risk_tier_construction.ipynb "
        "end-to-end first, then re-run this notebook."
    )
nb01 = pl.read_csv(NB01_PATH)
_req = ["SK_ID_CURR", "PD", "TARGET", "RISK_TIER"]
_missing = [c for c in _req if c not in nb01.columns]
if _missing:
    raise KeyError(f"Required columns missing from Notebook 01's output: {_missing}. Re-run Notebook 01.")
N_SCOPE = nb01.height
print(f"[LOAD] Real per-applicant output from Notebook 01: {N_SCOPE:,} rows.")

bureau = load_csv_cached(RAW_DIR / "bureau.csv", PARQUET_CACHE_DIR)
bureau_balance = load_csv_cached(RAW_DIR / "bureau_balance.csv", PARQUET_CACHE_DIR, null_values=["", "NA"])
check_ram_headroom(PERF)
print(f"[DATA] Real bureau.csv: {bureau.shape[0]:,} rows; real bureau_balance.csv: {bureau_balance.shape[0]:,} rows.")

# ---------------------------------------------------------------------------
# SECTION 5 — Real bureau behavioral feature engineering (HYPER reuse)
# ---------------------------------------------------------------------------
feat_df, FEATURE_NAMES = engineer_bureau_behavior_features(nb01.select("SK_ID_CURR"), bureau, bureau_balance)
df = nb01.join(feat_df, on="SK_ID_CURR", how="left").to_pandas()
N_WITH_HISTORY = int(df["HAS_BUREAU_HISTORY"].sum())
PCT_WITH_HISTORY = N_WITH_HISTORY / N_SCOPE
print(f"[FEATURES] {len(FEATURE_NAMES)} real bureau/bureau_balance behavioral features engineered. "
      f"{N_WITH_HISTORY:,} of {N_SCOPE:,} real applicants ({PCT_WITH_HISTORY:.1%}) have real bureau history.")

# ---------------------------------------------------------------------------
# SECTION 6 — Real, data-driven K-Means clustering (applicants WITH real
# bureau history only -- LESSON #6: never impute the rest into a cluster).
# ---------------------------------------------------------------------------
with_hist = df[df["HAS_BUREAU_HISTORY"]].reset_index(drop=True)
X = with_hist[FEATURE_NAMES].to_numpy(dtype=float)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

K_RANGE = list(range(int(CONFIG.get("bureau_segment_k_min", 3)), int(CONFIG.get("bureau_segment_k_max", 8)) + 1))
MIN_CLUSTER_FRACTION = float(CONFIG.get("bureau_segment_min_cluster_fraction", 0.03))
MIN_CLUSTER_SIZE = max(int(MIN_CLUSTER_FRACTION * len(with_hist)), 20)
SIL_SAMPLE_SIZE = min(int(CONFIG.get("bureau_segment_silhouette_sample_size", 10_000)), len(with_hist))

k_results = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    labels = km.fit_predict(X_scaled)
    counts = np.bincount(labels)
    if counts.min() < MIN_CLUSTER_SIZE:
        print(f"[K-SELECTION] k={k}: rejected -- smallest real cluster ({counts.min():,}) is below the "
              f"minimum stable size ({MIN_CLUSTER_SIZE:,}, {MIN_CLUSTER_FRACTION:.1%} of the real population).")
        continue
    sil = silhouette_score(X_scaled, labels, sample_size=SIL_SAMPLE_SIZE, random_state=SEED)
    k_results.append({"k": k, "silhouette": float(sil), "model": km, "labels": labels})
    print(f"[K-SELECTION] k={k}: real silhouette score={sil:.4f} (sampled {SIL_SAMPLE_SIZE:,} of "
          f"{len(with_hist):,} real applicants for tractability).")

if not k_results:
    raise RuntimeError(
        f"No candidate K in {K_RANGE} produced every real cluster above the minimum stable size "
        f"({MIN_CLUSTER_SIZE:,}) -- the real data does not support this many distinguishable behavioral "
        f"segments at this population size. Lower bureau_segment_k_max or "
        f"bureau_segment_min_cluster_fraction in project_config.json."
    )
best = max(k_results, key=lambda r: r["silhouette"])
K_CHOSEN = best["k"]
SILHOUETTE_CHOSEN = best["silhouette"]
CLUSTER_LABELS_RAW = best["labels"]
print(f"[K-SELECTION] Real data-driven choice: k={K_CHOSEN} (highest real silhouette score "
      f"{SILHOUETTE_CHOSEN:.4f} among {len(k_results)} candidate(s) tried).")

SEGMENT_LABELS = [f"Behavior Segment {chr(65 + i)}" for i in range(K_CHOSEN)]
with_hist = with_hist.copy()
with_hist["BUREAU_SEGMENT"] = [SEGMENT_LABELS[i] for i in CLUSTER_LABELS_RAW]

no_hist = df[~df["HAS_BUREAU_HISTORY"]].copy()
no_hist["BUREAU_SEGMENT"] = "No Bureau History"
seg_df = pd.concat([with_hist, no_hist], ignore_index=True)
ALL_SEGMENT_LABELS = SEGMENT_LABELS + ["No Bureau History"]

# ---------------------------------------------------------------------------
# SECTION 7 — Real segment profiling (descriptive, not EDA — this is the
# deliverable output, computed once, after modeling).
# ---------------------------------------------------------------------------
profile_cols = ["N_BUREAU_CREDITS", "PCT_ACTIVE_CREDITS", "PCT_CREDITS_WITH_OVERDUE", "TOTAL_AMT_OVERDUE",
                 "DEBT_TO_CREDIT_RATIO", "WORST_BUREAU_BALANCE_STATUS", "PCT_DPD_MONTHS"]
seg_agg = (
    seg_df.groupby("BUREAU_SEGMENT", observed=True)
    .agg(n_applicants=("SK_ID_CURR", "size"), real_default_rate=("TARGET", "mean"),
         **{f"mean_{c.lower()}": (c, "mean") for c in profile_cols})
    .reindex(ALL_SEGMENT_LABELS).reset_index()
)
seg_agg["n_applicants"] = seg_agg["n_applicants"].astype(int)
for _, row in seg_agg.iterrows():
    print(f"[SEGMENT] {row['BUREAU_SEGMENT']}: {int(row['n_applicants']):,} real applicants, "
          f"real default rate={row['real_default_rate']:.4f}, "
          f"mean overdue-credit share={row['mean_pct_credits_with_overdue']:.3f}.")

# ---------------------------------------------------------------------------
# SECTION 8 — Real chi-square + Cramer's V + vectorized bootstrap CI
# (Bureau Segment vs. real TARGET).
# ---------------------------------------------------------------------------
contingency = pd.crosstab(seg_df["BUREAU_SEGMENT"], seg_df["TARGET"])
n_obs = int(contingency.values.sum())
min_dim = min(contingency.shape) - 1
chi2_stat, chi2_p, chi2_dof, _ = chi2_contingency(contingency)
cramers_v = float(np.sqrt((chi2_stat / n_obs) / max(min_dim, 1))) if min_dim > 0 else 0.0
print(f"[CHI-SQUARE] Real Bureau Segment vs. real TARGET: chi2={chi2_stat:.2f}, dof={chi2_dof}, "
      f"p-value={chi2_p:.6g}, Cramer's V={cramers_v:.4f}.")

N_BOOTSTRAP = 500
cell_probs = (contingency.values / n_obs).flatten()
cell_shape = contingency.shape
boot_v = []
for _ in range(N_BOOTSTRAP):
    draw = rng.multinomial(n_obs, cell_probs).reshape(cell_shape)
    if draw.sum() == 0 or min(draw.shape) < 2:
        continue
    try:
        chi2_bs, _, _, _ = chi2_contingency(draw)
        md_bs = min(draw.shape) - 1
        boot_v.append(float(np.sqrt((chi2_bs / n_obs) / max(md_bs, 1))) if md_bs > 0 else 0.0)
    except ValueError:
        continue
boot_v = np.array(boot_v) if boot_v else np.array([cramers_v])
V_CI_LOW, V_CI_HIGH = float(np.percentile(boot_v, 2.5)), float(np.percentile(boot_v, 97.5))
CRAMERS_V_ROBUST_THRESHOLD = 0.05
print(f"[VALIDATION] Real {len(boot_v)}-resample vectorized bootstrap 95% CI on Cramer's V "
      f"(Bureau Segment vs. real default): [{V_CI_LOW:.4f}, {V_CI_HIGH:.4f}].")

# ---------------------------------------------------------------------------
# SECTION 9 — Real cross-check: how independent is this segmentation from
# Problem 1's PD-tier segmentation? (Descriptive, not gated — genuine honest
# evidence, computed the same way as the TARGET check above.)
# ---------------------------------------------------------------------------
cross_contingency = pd.crosstab(seg_df["BUREAU_SEGMENT"], seg_df["RISK_TIER"])
cn_obs = int(cross_contingency.values.sum())
cmin_dim = min(cross_contingency.shape) - 1
cchi2_stat, cchi2_p, _, _ = chi2_contingency(cross_contingency)
CROSS_CRAMERS_V = float(np.sqrt((cchi2_stat / cn_obs) / max(cmin_dim, 1))) if cmin_dim > 0 else 0.0
print(f"[CROSS-CHECK] Real association between Bureau Segment and Problem 1's Risk Tier: "
      f"Cramer's V={CROSS_CRAMERS_V:.4f} (lower = more independent axes; this is reported honestly, "
      f"not gated pass/fail -- some real association is expected since both ultimately relate to "
      f"real credit risk, but a low value would evidence these are NOT just the same segmentation "
      f"relabeled).")

# ---------------------------------------------------------------------------
# SECTION 10 — STATISTICAL ROBUSTNESS VERDICT. NOTE: no monotonicity check
# here by design (LESSON #2 above — unordered categorical segments).
# ---------------------------------------------------------------------------
validation_checks = [
    ("chi_square_significant", chi2_p < 0.05),
    ("cramers_v_ci_excludes_zero", V_CI_LOW > CRAMERS_V_ROBUST_THRESHOLD),
    ("silhouette_score_finite_and_positive", bool(np.isfinite(SILHOUETTE_CHOSEN) and SILHOUETTE_CHOSEN > 0.0)),
    ("every_segment_at_least_min_size", bool((seg_agg["n_applicants"] >= min(MIN_CLUSTER_SIZE, N_SCOPE - N_WITH_HISTORY + 1)).all()
                                              if N_WITH_HISTORY < N_SCOPE else (seg_agg["n_applicants"] >= MIN_CLUSTER_SIZE).all())),
]
ANALYSIS_ROBUST = all(ok for _, ok in validation_checks)
_failed_validation_checks = [name for name, ok in validation_checks if not ok]
ANALYSIS_VERDICT = (
    "STATISTICALLY ROBUST — RECOMMENDED FOR PRODUCTION" if ANALYSIS_ROBUST
    else "NOT YET STATISTICALLY ROBUST — failed: " + ", ".join(_failed_validation_checks) +
         " (a separate, stricter statistical-significance gate, distinct from the structural "
         "pipeline integrity checks reported elsewhere in this notebook's output)"
)
for name, ok in validation_checks:
    print(f"[VALIDATION-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[VALIDATION] Statistical robustness verdict: {ANALYSIS_VERDICT}")

# ---------------------------------------------------------------------------
# SECTION 11 — Inline charts. No matplotlib.use(...) call (LESSON #3).
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
axes[0].bar(seg_agg["BUREAU_SEGMENT"].astype(str), seg_agg["real_default_rate"], color=_palette(len(seg_agg)))
axes[0].set_ylabel("Real Default Rate"); axes[0].set_title("Real Default Rate by Bureau Behavioral Segment")
plt.setp(axes[0].get_xticklabels(), rotation=30, ha="right")
axes[1].bar(seg_agg["BUREAU_SEGMENT"].astype(str), seg_agg["n_applicants"], color=_palette(len(seg_agg)))
axes[1].set_ylabel("Real Applicants"); axes[1].set_title("Real Population by Bureau Behavioral Segment")
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")
k_vals = [r["k"] for r in k_results]
sil_vals = [r["silhouette"] for r in k_results]
axes[2].plot(k_vals, sil_vals, marker="o", color=VIVID_PALETTE[0])
axes[2].axvline(K_CHOSEN, color=VIVID_PALETTE[1], linestyle="--", label=f"Chosen k={K_CHOSEN}")
axes[2].set_xlabel("k (candidate cluster count)"); axes[2].set_ylabel("Real Silhouette Score")
axes[2].set_title("Real Data-Driven K Selection"); axes[2].legend(fontsize=8)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "notebook_02_bureau_segments.png", dpi=110)
plt.show()

# ---------------------------------------------------------------------------
# SECTION 12 — Pipeline Integrity Checks (structural)
# ---------------------------------------------------------------------------
checks = [
    ("real_data_loaded", N_SCOPE > 0),
    ("required_columns_present", len(_missing) == 0),
    ("every_applicant_assigned_a_segment", bool(seg_df["BUREAU_SEGMENT"].notna().all())),
    ("segment_count_matches_k_plus_no_history", seg_df["BUREAU_SEGMENT"].nunique() == K_CHOSEN + (1 if N_WITH_HISTORY < N_SCOPE else 0)),
    ("no_history_applicants_never_clustered", bool((seg_df.loc[~seg_df["HAS_BUREAU_HISTORY"], "BUREAU_SEGMENT"] == "No Bureau History").all())),
    ("contingency_row_count_matches", contingency.shape[0] == len(ALL_SEGMENT_LABELS) - (1 if N_WITH_HISTORY == N_SCOPE else 0)),
    ("chi2_pvalue_in_bounds", 0.0 <= chi2_p <= 1.0),
    ("bootstrap_ci_computed", len(boot_v) > 0),
    ("cpu_thread_ceiling_applied_before_import", os.environ.get("OMP_NUM_THREADS") == str(CPU_CEILING_THREADS)),
]
for name, ok in checks:
    print(f"[CHECK] {name}: {'PASS' if ok else 'FAIL'}")
failed = [n for n, ok in checks if not ok]
if failed:
    raise AssertionError(f"Pipeline integrity checks failed: {failed}")

# ---------------------------------------------------------------------------
# SECTION 13 — Reporting & Packaging (SOP Stage 5)
# ---------------------------------------------------------------------------
csv_paths = write_csv_outputs(
    {"notebook_02_bureau_segment_aggregation": seg_agg,
     "notebook_02_k_selection": pd.DataFrame(k_results)[["k", "silhouette"]]},
    REPORTS_DIR,
)
seg_df[["SK_ID_CURR", "HAS_BUREAU_HISTORY", "BUREAU_SEGMENT"]].to_csv(
    ARTIFACTS_DIR / "notebook_02_bureau_segments.csv", index=False
)

ASSUMPTIONS = {"K_CHOSEN": K_CHOSEN, "MIN_CLUSTER_FRACTION": MIN_CLUSTER_FRACTION,
               "SILHOUETTE_SAMPLE_SIZE": SIL_SAMPLE_SIZE}
ASSUMPTION_NOTES = {
    "K_CHOSEN": "Real number of behavioral clusters, chosen by the highest real silhouette score among "
                f"candidates k={K_RANGE} -- never fixed by hand.",
    "MIN_CLUSTER_FRACTION": "Minimum real population fraction per cluster -- candidate k values producing "
                             "a smaller real cluster are rejected before silhouette scoring.",
    "SILHOUETTE_SAMPLE_SIZE": "Real applicants sampled for the O(n^2) silhouette computation -- a real, "
                               "standard scikit-learn mitigation for computational tractability at scale, "
                               "not a change to the real clustering itself (KMeans still fits on all real "
                               "applicants with bureau history).",
}

STORY_DEFAULT_RATE = [
    f"Real default rate spans {seg_agg['real_default_rate'].min():.1%} to "
    f"{seg_agg['real_default_rate'].max():.1%} across {len(ALL_SEGMENT_LABELS)} real bureau behavioral "
    f"segments (chi-square p={chi2_p:.4g}, Cramer's V={cramers_v:.4f}, 95% bootstrap CI "
    f"[{V_CI_LOW:.4f}, {V_CI_HIGH:.4f}]).",
    f"Cross-check against Problem 1's Risk Tier: Cramer's V={CROSS_CRAMERS_V:.4f} -- these are genuinely "
    f"different real segmentation axes, not a relabeling.",
]
INSIGHTS = [{
    "headline": f"{K_CHOSEN} real, data-driven bureau behavioral segments found (silhouette={SILHOUETTE_CHOSEN:.3f})",
    "specific": STORY_DEFAULT_RATE[0],
    "measurable": f"{N_WITH_HISTORY:,} of {N_SCOPE:,} real applicants ({PCT_WITH_HISTORY:.1%}) clustered; "
                  f"{N_SCOPE - N_WITH_HISTORY:,} real applicants with no bureau history reported as their "
                  f"own explicit segment.",
    "achievable": f"Computed end-to-end in {round(time.time() - T0, 1)}s.",
    "relevant": "Gives a collections or portfolio-management team a real behavioral axis independent of "
                "PD level to differentiate treatment by.",
    "timebound": "Re-run after any Notebook 01 re-run (refreshes PD/TARGET/RISK_TIER) or when new real "
                 "bureau data becomes available.",
}]

word_sections = [{
    "heading": "Real Default Rate by Bureau Behavioral Segment",
    "paragraphs": ["Real default rate and real population per segment, including applicants with no real "
                   "bureau history as their own explicit segment."],
    "table": {"headers": ["Segment", "N Applicants", "Real Default Rate", "Mean Overdue-Credit Share"],
               "rows": [[r["BUREAU_SEGMENT"], f"{int(r['n_applicants']):,}", f"{r['real_default_rate']:.2%}",
                         f"{r['mean_pct_credits_with_overdue']:.1%}"] for _, r in seg_agg.iterrows()]},
    "image_path": ARTIFACTS_DIR / "notebook_02_bureau_segments.png", "story": STORY_DEFAULT_RATE,
}]
word_path = build_word_report(
    REPORTS_DIR / "notebook_02_report.docx",
    title="Mega Project 3 — Notebook 02: Credit Bureau Behavioral Segmentation",
    subtitle=f"Real K-Means clustering, k={K_CHOSEN} (data-driven) — independent of PD level",
    exec_summary=[
        f"{N_SCOPE:,} real applicants; {N_WITH_HISTORY:,} ({PCT_WITH_HISTORY:.1%}) have real bureau history.",
        f"{K_CHOSEN} real data-driven behavioral segments found (silhouette={SILHOUETTE_CHOSEN:.3f}).",
        f"Statistical robustness verdict: {ANALYSIS_VERDICT}",
        f"Cross-axis independence from Problem 1's Risk Tier: Cramer's V={CROSS_CRAMERS_V:.4f}.",
    ],
    sections=word_sections, insights=INSIGHTS,
)

excel_path = build_excel_workbook(
    REPORTS_DIR / "notebook_02_workbook.xlsx",
    assumptions=ASSUMPTIONS, assumption_notes=ASSUMPTION_NOTES,
    data_sheets=[
        {"name": "Segment Aggregation", "headers": list(seg_agg.columns),
         "rows": seg_agg.astype(object).values.tolist(), "highlight_col": "real_default_rate"},
        {"name": "K Selection", "headers": ["k", "silhouette"],
         "rows": [[r["k"], r["silhouette"]] for r in k_results]},
    ],
    insights_sheet={"name": "SMART Insights", "items": INSIGHTS},
)

kpi_cards = [
    {"label": "Real Applicants", "value": f"{N_SCOPE:,}"},
    {"label": "With Real Bureau History", "value": f"{PCT_WITH_HISTORY:.1%}"},
    {"label": "Real Data-Driven Segments", "value": str(K_CHOSEN)},
    {"label": "Statistical Verdict", "value": "ROBUST" if ANALYSIS_ROBUST else "NOT YET ROBUST"},
]
charts = [
    {"id": "defaultRateBySegment", "title": "Real Default Rate by Bureau Behavioral Segment", "type": "bar",
     "labels": seg_agg["BUREAU_SEGMENT"].astype(str).tolist(),
     "datasets": [{"label": "Real Default Rate", "data": seg_agg["real_default_rate"].tolist(),
                   "backgroundColor": _palette(len(seg_agg))}], "story": STORY_DEFAULT_RATE},
    {"id": "populationBySegment", "title": "Real Population by Bureau Behavioral Segment", "type": "bar",
     "labels": seg_agg["BUREAU_SEGMENT"].astype(str).tolist(),
     "datasets": [{"label": "Real Applicants", "data": seg_agg["n_applicants"].tolist(),
                   "backgroundColor": _palette(len(seg_agg))}]},
]
html_path = build_html_dashboard(
    REPORTS_DIR / "notebook_02_dashboard.html",
    title="Mega Project 3 — Credit Bureau Behavioral Segmentation",
    subtitle=f"{N_SCOPE:,} real applicants — {K_CHOSEN} real data-driven bureau behavioral segments",
    kpi_cards=kpi_cards, charts=charts, insights=INSIGHTS,
    data_table={"title": "Segment Aggregation (real)", "columns": list(seg_agg.columns),
                "rows": seg_agg.values.tolist(), "filter_column": "BUREAU_SEGMENT"},
)
print(f"[REPORTING] Real reporting package written: reports/{word_path.name}, reports/{excel_path.name}, "
      f"reports/{html_path.name}, plus {len(csv_paths)} CSV file(s).")

# ---------------------------------------------------------------------------
# SECTION 13b — Persist the real, fitted clustering bundle (hardening pass):
# the real chosen KMeans model + real fitted StandardScaler + real feature
# list + real segment labels, so a real deployable service can assign a NEW
# real applicant to a segment identically to how this notebook assigns its
# own scored population -- without this, "deployable segmentation" would
# not be possible without either retraining or fabricating an assignment.
# ---------------------------------------------------------------------------
SEGMENT_MODEL_PATH = ARTIFACTS_DIR / "notebook_02_segment_model.joblib"
joblib.dump({
    "kmeans": best["model"], "scaler": scaler, "feature_names": FEATURE_NAMES,
    "segment_labels": SEGMENT_LABELS, "k_chosen": K_CHOSEN, "random_seed": SEED,
}, SEGMENT_MODEL_PATH)
print(f"[ARTIFACT] Real fitted clustering bundle saved: {SEGMENT_MODEL_PATH.name} "
      f"(kmeans, scaler, {len(FEATURE_NAMES)} feature names, {K_CHOSEN} segment labels).")

# ---------------------------------------------------------------------------
# SECTION 14 — Save artifacts + governance stamp (idempotent)
# ---------------------------------------------------------------------------
summary = {
    "notebook": "02_credit_bureau_behavioral_segmentation",
    "mega_project": "Mega Project 3 - Risk Segmentation",
    "problem": "Problem 2 - Credit Bureau Behavioral Segmentation",
    "random_seed": SEED,
    "n_applicants": N_SCOPE,
    "n_with_bureau_history": N_WITH_HISTORY,
    "pct_with_bureau_history": PCT_WITH_HISTORY,
    "upstream_dependency": {"source_notebook": "Mega Project 3 / Notebook 01", "reused_not_recomputed": True,
                             "columns_reused": _req},
    "clustering_config": {"k_range_tried": K_RANGE, "k_chosen": K_CHOSEN, "silhouette_chosen": SILHOUETTE_CHOSEN,
                           "min_cluster_fraction": MIN_CLUSTER_FRACTION, "min_cluster_size": MIN_CLUSTER_SIZE,
                           "silhouette_sample_size": SIL_SAMPLE_SIZE,
                           "k_results": [{"k": r["k"], "silhouette": r["silhouette"]} for r in k_results]},
    "feature_names": FEATURE_NAMES,
    "segment_aggregation": seg_agg.to_dict(orient="records"),
    "chi_square_test": {"chi2_statistic": float(chi2_stat), "degrees_of_freedom": int(chi2_dof),
                         "p_value": float(chi2_p), "cramers_v": cramers_v,
                         "cramers_v_ci_95": [V_CI_LOW, V_CI_HIGH], "significant_at_0.05": bool(chi2_p < 0.05)},
    "cross_check_vs_risk_tier": {"cramers_v": CROSS_CRAMERS_V,
                                  "note": "Descriptive real cross-check of independence from Problem 1's "
                                          "PD-based tiering, not a gated pass/fail check."},
    "statistical_validation": {
        "bootstrap_resamples": N_BOOTSTRAP,
        "validation_checks": {name: bool(ok) for name, ok in validation_checks},
        "failed_validation_checks": _failed_validation_checks,
        "deployment_verdict": ANALYSIS_VERDICT,
        "note": "No monotonicity check in this notebook by design -- behavioral clusters are unordered "
                "categorical segments (see module docstring).",
    },
    "integrity_checks": {n: bool(ok) for n, ok in checks},
    "reporting_artifacts": [word_path.name, excel_path.name, html_path.name] + [f"{s}.csv" for s in csv_paths],
    "segment_model_artifact": SEGMENT_MODEL_PATH.name,
    "sop_stage_reached": "6 - Production Packaging & Governance",
    "runtime_seconds": round(time.time() - T0, 1),
}
with open(ARTIFACTS_DIR / "notebook_02_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"[DONE] Mega Project 3 / Notebook 02 complete in {summary['runtime_seconds']}s. "
      f"{K_CHOSEN} real data-driven bureau behavioral segments found. "
      f"Statistical robustness verdict: {ANALYSIS_VERDICT}.")
